In [ ]:
# imports
import torch
import shutil
import numpy as np
from torch import nn
from pathlib import Path
from kagglehub import dataset_download
from sklearn import linear_model, neighbors
from sklearn.neighbors import KNeighborsClassifier
from torchvision import transforms, datasets
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset, DataLoader, WeightedRandomSampler

In [6]:
# download dataset

local_data_path = Path("../data")
dataset_base_path = Path(dataset_download("crawford/cat-dataset"))
dataset_base_path = dataset_base_path / "cats"
oreo_path = local_data_path / "oreo"
not_oreo_path = local_data_path / "not_oreo"

oreo_path.mkdir(exist_ok=True)
not_oreo_path.mkdir(exist_ok=True)

for image in local_data_path.glob("*.jpg"):
    shutil.move(str(image), str(oreo_path / image.name))

for subdir in dataset_base_path.iterdir():
    if subdir.is_dir():
        for image in subdir.glob("*.jpg"):
            new_name = f"{subdir.name}_{image.name}"
            shutil.move(str(image), str(not_oreo_path / new_name))

In [7]:
# preprocessing

# Resize, Augment, and Normalize RGB
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2), # random brightness
    transforms.ToTensor()
])

full_dataset = datasets.ImageFolder(root=local_data_path, transform=transform)
train_loader = DataLoader(full_dataset, batch_size=32, shuffle=True)

In [8]:
# Split

targets = np.array(full_dataset.targets)
indices = np.arange(len(full_dataset))

# 80% Train
train_idx, temp_idx = train_test_split(
    indices,
    test_size=0.2,
    stratify=targets,
    random_state=42
)

# 10% validation
# 10% test
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.5,
    stratify=targets[temp_idx],
    random_state=42
)

train_dataset = Subset(full_dataset, train_idx)
val_dataset = Subset(full_dataset, val_idx)
test_dataset = Subset(full_dataset, test_idx)

In [9]:
# Weighted Random Sampler

train_targets = targets[train_idx]
class_sample_count = np.array([len(np.where(train_targets == t)[0]) for t in np.unique(train_targets)])
weight = 1. / class_sample_count
samples_weight = np.array([weight[t] for t in train_targets])
samples_weight = torch.from_numpy(samples_weight)

sampler = WeightedRandomSampler(
    weights=samples_weight,
    num_samples=len(samples_weight),
    replacement=True
)

train_loader = DataLoader(train_dataset, batch_size=32, sampler=sampler)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

images, labels = next(iter(train_loader))
oreo_count = (labels == full_dataset.class_to_idx['oreo']).sum().item()
not_oreo_count = (labels == full_dataset.class_to_idx['not_oreo']).sum().item()

In [ ]:
# logistic Regression

In [ ]:
# K-NN
x_train = []
y_train = []

for images, labels in train_loader:
    # Flattening images and converting to numpy
    x_train.append(images.view(images.size(0), -1).numpy())
    y_train.append(labels.numpy())

# Creating arrays for training
x_train = np.vstack(x_train)
y_train = np.concatenate(y_train)

# Training
neighbors = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
neighbors.fit(x_train, y_train)

# Testing
test_images, test_labels = next(iter(test_loader))
x_test = test_images.view(test_images.size(0), -1).numpy()

# Printing predictions and actual labels
predictions = neighbors.predict(x_test)
print(f"Predictions: {predictions}")
print(f"Actual labels: {test_labels.numpy()}")

In [ ]:
# CNN

In [ ]:
# evaluation